In [1]:
import os

In [2]:
%pwd

'/Users/admin/PycharmProjects/AnimalsDetectionMLOPS/notebooks'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/Users/admin/PycharmProjects/AnimalsDetectionMLOPS'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class PrepareCallbacksConfig:
    """Configuration for preparing training callbacks."""
    root_dir: Path
    checkpoint_model_filepath: Path
    tensorboard_log_dir: Path

In [6]:
@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list

In [7]:
from src.cnnClassifier.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from src.cnnClassifier.utils.common import read_yaml, create_directories
import torch
from torchvision import models

In [8]:
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_prepare_callbacks_config(self) -> PrepareCallbacksConfig:
        config = self.config.prepare_callbacks
        create_directories([config.root_dir])
        return PrepareCallbacksConfig(
            root_dir=Path(config.root_dir),
            checkpoint_model_filepath=Path(config.checkpoint_model_filepath),
            tensorboard_log_dir=Path(config.tensorboard_root_log_dir),
        )

    def get_training_config(self)-> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params
        # Use the data_ingestion root as ImageFolder root (expects class subfolders)
        training_data = Path(self.config.data_ingestion.unzip_dir)
        create_directories([
            Path(training.root_dir)
        ])
        training_config = TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            training_data=Path(training_data),
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE
        )

        return training_config


In [9]:
import time

In [10]:
class CheckpointCallback:
    """Callback that saves model state dict per epoch."""

    def __init__(self, filepath: Path):
        self.filepath = filepath

    def save(self, model, epoch: int):
        torch.save(model.state_dict(), str(self.filepath).format(epoch=epoch))

In [11]:

from torch.utils.tensorboard import SummaryWriter


class PrepareCallbacks:
    """Factory for preparing TensorBoard and checkpoint callbacks."""

    def __init__(self, config: PrepareCallbacksConfig):
        self.config = config
        create_directories([
            self.config.root_dir,
            self.config.tensorboard_log_dir.parent,
            self.config.checkpoint_model_filepath.parent,
        ])

    def get_tensorboard_callback(self) -> SummaryWriter:
        """Return a torch SummaryWriter pointing to the tensorboard log dir."""
        return SummaryWriter(log_dir=str(self.config.tensorboard_log_dir))

    def get_checkpoint_callback(self) -> CheckpointCallback:
        """Return a checkpoint callback that saves model state dict per epoch."""
        return CheckpointCallback(filepath=self.config.checkpoint_model_filepath)

In [12]:
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

In [13]:
class Training:
    def __init__(self, config: TrainingConfig,
                 tensorboard_callback: SummaryWriter,
                 checkpoint_callback: CheckpointCallback):
        self.config = config
        self.tensorboard_callback = tensorboard_callback
        self.checkpoint_callback = checkpoint_callback

        self.device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
        self.model = self._load_model().to(self.device)
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=0.001)
        self.train_loader = self._get_data_loader()

    def _load_model(self):
        # Start from pretrained VGG16 features
        model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        # Determine number of classes from training data structure
        try:
            class_dirs = [d for d in os.listdir(self.config.training_data) if (self.config.training_data / d).is_dir()]
            num_classes = len(class_dirs) if len(class_dirs) > 0 else 2
        except Exception:
            num_classes = 2
        # Replace the final classifier layer to match dataset classes
        in_features = model.classifier[6].in_features
        model.classifier[6] = nn.Linear(in_features, num_classes)
        # Optionally load a compatible PyTorch checkpoint if present
        path = self.config.updated_base_model_path
        if str(path).endswith('.pth') or str(path).endswith('.pt'):
            try:
                from torch.serialization import add_safe_globals
                add_safe_globals([models.vgg.VGG])
            except Exception:
                pass
            try:
                state_obj = torch.load(path, weights_only=True, map_location=self.device)
                sd = state_obj.state_dict() if hasattr(state_obj, 'state_dict') else state_obj
                model.load_state_dict(sd, strict=False)
            except Exception:
                try:
                    state_obj = torch.load(path, weights_only=False, map_location=self.device)
                    sd = state_obj.state_dict() if hasattr(state_obj, 'state_dict') else state_obj
                    model.load_state_dict(sd, strict=False)
                except Exception:
                    pass
        return model

    def _get_data_loader(self):
        transform_list = [transforms.Resize((self.config.params_image_size[0], self.config.params_image_size[1])),
                          transforms.ToTensor()]
        if self.config.params_is_augmentation:
            transform_list.insert(0, transforms.RandomHorizontalFlip())

        transform = transforms.Compose(transform_list)

        dataset = datasets.ImageFolder(root=str(self.config.training_data), transform=transform)
        dataloader = DataLoader(dataset, batch_size=self.config.params_batch_size, shuffle=True)
        return dataloader
    @staticmethod
    def save_model(model, path: Path):
        torch.save(model.state_dict(), str(path))
    def train(self):
        for epoch in range(self.config.params_epochs):
            epoch_loss = 0.0
            for inputs, labels in self.train_loader:
                inputs, labels = inputs.to(self.device), labels.to(self.device)

                self.optimizer.zero_grad()
                outputs = self.model(inputs)
                loss = self.criterion(outputs, labels)
                loss.backward()
                self.optimizer.step()

                epoch_loss += loss.item() * inputs.size(0)

            epoch_loss /= len(self.train_loader.dataset)
            self.tensorboard_callback.add_scalar('Loss/train', epoch_loss, epoch)
            print(f'Epoch {epoch+1}/{self.config.params_epochs}, Loss: {epoch_loss:.4f}')

            # Save checkpoint
            self.checkpoint_callback.save(self.model, epoch+1)

        # Save the final trained model
        self.save_model(self.model, self.config.trained_model_path)

In [14]:
try:
    config_manager = ConfigurationManager()
    prepare_callbacks_config = config_manager.get_prepare_callbacks_config()
    training_config = config_manager.get_training_config()

    prepare_callbacks = PrepareCallbacks(config=prepare_callbacks_config)

    tensorboard_callback = prepare_callbacks.get_tensorboard_callback()
    checkpoint_callback = prepare_callbacks.get_checkpoint_callback()

    training = Training(
        config=training_config,
        tensorboard_callback=tensorboard_callback,
        checkpoint_callback=checkpoint_callback
    )

    training.train()

    print(f"Trained model saved at: {training_config.trained_model_path}")

except Exception as e:
    print(f"Error during training: {e}")

[2026-01-22 13:39:12,316: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-01-22 13:39:12,317: INFO: common: yaml file: params.yaml loaded successfully]
[2026-01-22 13:39:12,318: INFO: common: created directory at: artifacts]
[2026-01-22 13:39:12,318: INFO: common: created directory at: artifacts/prepare_callbacks]
[2026-01-22 13:39:12,318: INFO: common: created directory at: artifacts/training]
[2026-01-22 13:39:12,319: INFO: common: created directory at: artifacts/prepare_callbacks]
[2026-01-22 13:39:12,320: INFO: common: created directory at: artifacts/prepare_callbacks]
[2026-01-22 13:39:12,321: INFO: common: created directory at: artifacts/prepare_callbacks/checkpoint_dir]
Epoch 1/1, Loss: 0.8059
Trained model saved at: artifacts/training/model.h5
